In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, log_loss

# reproducibility
SEED = 42
np.random.seed(SEED)

# display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Data Pipeline

In [2]:
# loading and preparing the ionosphere dataset
data = fetch_openml(name="ionosphere", version=1, as_frame=True)

X = data.data.to_numpy(dtype=float)
y = data.target.to_numpy()

y = np.where(y == "g", 1, 0)

print("Dataset shape:", X.shape)
print("Class distribution:", np.bincount(y))

Dataset shape: (351, 34)
Class distribution: [126 225]


In [3]:
# perform a 70/30 split to create a training and test set
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=SEED,
    stratify=y
)

print("Training pool shape:", X_train_full.shape)
print("Test shape:", X_test.shape)

Training pool shape: (245, 34)
Test shape: (106, 34)


# Model Implementation

## Logistic Regression

In [4]:
# training function for logistic regression
def train_logistic(X_train, y_train):
    model = LogisticRegression(
        max_iter=5000,
        solver="liblinear",
        fit_intercept=True,
        random_state=SEED
    )
    model.fit(X_train, y_train)
    return model

# evaluation function for logistic regression
def evaluate_logistic(model, X_train, y_train, X_test, y_test):
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    y_train_prob = model.predict_proba(X_train)
    y_test_prob = model.predict_proba(X_test)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    # cross-entropy loss
    train_loss = log_loss(y_train, y_train_prob)
    test_loss = log_loss(y_test, y_test_prob)

    # generalization gap
    gap = train_acc - test_acc

    return train_acc, test_acc, train_loss, test_loss, gap

## LDA

In [5]:
# training function for linear discriminant analysis
def train_lda(X_train, y_train):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        model = LinearDiscriminantAnalysis(solver="eigen", shrinkage="auto")
        model.fit(X_train, y_train)
    return model

# evaluation function for linear discriminant analysis
def evaluate_lda(model, X_train, y_train, X_test, y_test):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        y_train_pred = model.predict(X_train)
        y_test_pred  = model.predict(X_test)
        y_train_prob = model.predict_proba(X_train)
        y_test_prob  = model.predict_proba(X_test)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc  = accuracy_score(y_test,  y_test_pred)
    train_loss = log_loss(y_train, y_train_prob)
    test_loss  = log_loss(y_test,  y_test_prob)
    gap = train_acc - test_acc

    return train_acc, test_acc, train_loss, test_loss, gap

# Experiments

## Varying Sample Size (N) and Measuring Performance

In [10]:
# ─────────────────────────────────────────────────────────────
# EXPERIMENT 1: Varying Sample Size (N), Fixed M=34
# ─────────────────────────────────────────────────────────────

sample_fracs = np.linspace(0.1, 1.0, 10)
results_n = []

for i, frac in enumerate(sample_fracs):
    n = max(int(frac * len(X_train_full)), 2)

    rng = np.random.default_rng(SEED + i)

    idx = []
    for cls in np.unique(y_train_full):
        cls_idx = np.where(y_train_full == cls)[0]
        k = max(1, round(n * len(cls_idx) / len(y_train_full)))
        k = min(k, len(cls_idx))
        idx.extend(rng.choice(cls_idx, size=k, replace=False))

    idx = np.array(idx)
    rng.shuffle(idx)

    X_sub = X_train_full[idx]
    y_sub = y_train_full[idx]

    scaler_n = StandardScaler()
    X_sub_sc = scaler_n.fit_transform(X_sub)
    X_test_sc_n = scaler_n.transform(X_test)

    lr = train_logistic(X_sub_sc, y_sub)
    lda = train_lda(X_sub_sc, y_sub)

    lr_metrics = evaluate_logistic(lr, X_sub_sc, y_sub, X_test_sc_n, y_test)
    lda_metrics = evaluate_lda(lda, X_sub_sc, y_sub, X_test_sc_n, y_test)

    results_n.append(dict(
        n=len(idx),
        M=X_sub_sc.shape[1],
        M_over_N=X_sub_sc.shape[1] / len(idx),

        lr_train_acc=lr_metrics[0],
        lr_test_acc=lr_metrics[1],
        lr_train_loss=lr_metrics[2],
        lr_test_loss=lr_metrics[3],
        lr_gap=lr_metrics[4],

        lda_train_acc=lda_metrics[0],
        lda_test_acc=lda_metrics[1],
        lda_train_loss=lda_metrics[2],
        lda_test_loss=lda_metrics[3],
        lda_gap=lda_metrics[4],
    ))

df_n = pd.DataFrame(results_n)
print("=== Varying N (stratified, reproducible) ===")
print(df_n.to_string(index=False))

=== Varying N (stratified, reproducible) ===
  n  M  M_over_N  lr_train_acc  lr_test_acc  lr_train_loss  lr_test_loss   lr_gap  lda_train_acc  lda_test_acc  lda_train_loss  lda_test_loss   lda_gap
 24 34  1.416667      1.000000     0.783019       0.085868      0.553651 0.216981       0.958333      0.764151        0.091227       1.362721  0.194182
 49 34  0.693878      0.979592     0.858491       0.098599      0.423205 0.121101       0.979592      0.858491        0.119960       0.636430  0.121101
 73 34  0.465753      0.986301     0.839623       0.140433      0.499397 0.146679       0.917808      0.867925        0.232298       0.399567  0.049884
 98 34  0.346939      0.989796     0.905660       0.103468      0.333676 0.084136       0.928571      0.886792        0.204540       0.423117  0.041779
122 34  0.278689      0.942623     0.896226       0.168666      0.314236 0.046397       0.926230      0.905660        0.276924       0.270560  0.020569
147 34  0.231293      0.972789     0.886792

## Varying Feature Dimension (M) and Measuring Performance

## Model Comparison under Fixed (M, N)

## Evaluation Metrics (Accuracy, Loss, Generalization Gap)

# Results & Visualization

## Performance vs Capacity Ratio (M/N)

## Generalization Gap across Models

## Behavior near Interpolation (M $\approx$ N)